In [13]:
import sys
sys.path.append('../src')
from feature_engineering import engineer_features

# Reload fresh cleaned data (not the already-encoded df from earlier cells)
df_raw = pd.read_csv('../data/processed/cleaned_car_data.csv')

df_encoded, freq_map = engineer_features(df_raw)
df_encoded.head()

,vehicle_age,km_driven,mileage,engine,max_power,seats,selling_price,model_freq,brand_BMW,brand_Datsun,...,brand_Tata,brand_Toyota,brand_Volkswagen,seller_type_Individual,seller_type_Trustmark Dealer,fuel_type_Diesel,fuel_type_Electric,fuel_type_LPG,fuel_type_Petrol,transmission_type_Manual
0,9,120000,19.70,796,46.30,5,120000,778,0,0,...,0,0,0,1,0,0,0,0,1,1
1,5,20000,18.90,1197,82.00,5,550000,580,0,0,...,0,0,0,1,0,0,0,0,1,1
2,11,60000,17.00,1197,80.00,5,215000,906,0,0,...,0,0,0,1,0,0,0,0,1,1
3,9,37000,20.92,998,67.10,5,226000,778,0,0,...,0,0,0,1,0,0,0,0,1,1
4,6,30000,22.77,1498,98.59,5,570000,373,0,0,...,0,0,0,0,0,1,0,0,0,1


In [14]:
import json

with open('../models/model_freq_map.json', 'w') as f:
    json.dump(freq_map, f, indent=4)

reference_columns = df_encoded.drop(columns=['selling_price']).columns.tolist()
with open('../models/reference_columns.json', 'w') as f:
    json.dump(reference_columns, f, indent=4)

print('Saved freq_map and reference_columns.')
print('Number of reference columns:', len(reference_columns))

Saved freq_map and reference_columns.
Number of reference columns: 29


In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/cleaned_car_data.csv')
df.head()

,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [2]:
df['model'].nunique()

111

In [3]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
for col in categorical_cols:
    print(col, '->', df[col].nunique())

brand -> 16
model -> 111
seller_type -> 3
fuel_type -> 5
transmission_type -> 2


C:\Users\Dell\AppData\Local\Temp\ipykernel_2380\1150669931.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns.tolist()


In [5]:

df['model_freq'] = df['model'].map(df['model'].value_counts())

In [6]:
df = pd.get_dummies(df, columns=['brand', 'seller_type', 'fuel_type', 'transmission_type'], drop_first=True)

In [7]:
df = df.drop(columns=['model'])
df.head()
df.shape

(15281, 30)

In [8]:
df.dtypes

vehicle_age                       int64
km_driven                         int64
mileage                         float64
engine                            int64
max_power                       float64
seats                             int64
selling_price                     int64
model_freq                        int64
brand_BMW                          bool
brand_Datsun                       bool
brand_Ford                         bool
brand_Honda                        bool
brand_Hyundai                      bool
brand_Jaguar                       bool
brand_Mahindra                     bool
brand_Maruti                       bool
brand_Mercedes-Benz                bool
brand_Other                        bool
brand_Renault                      bool
brand_Skoda                        bool
brand_Tata                         bool
brand_Toyota                       bool
brand_Volkswagen                   bool
seller_type_Individual             bool
seller_type_Trustmark Dealer       bool


In [9]:
df.to_csv('../data/processed/featured_car_data.csv', index=False)

In [10]:
X = df.drop(columns=['selling_price'])
y = df['selling_price']

In [11]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)